In [20]:
import tensorflow as tf
import pandas as pd
import numpy as np
data = pd.read_csv("data.csv")
data = data[['R','G', 'B', 'L_cal', 'a_cal', 'b_cal', 'L*', 'a*', 'b*', 'VDO', 'File_Name', 'Crop_Index']]

# 1. normalize 'R', 'G', 'B' (min = 0, max = 255) to [0,1]
data[['R', 'G', 'B']] = data[['R', 'G', 'B']] / 255.0

# 2. normalize 'L_cal', 'L*' (min = 0, max = 100) to [0,1]
data[['L_cal', 'L*']] = data[['L_cal', 'L*']] / 100.0

# 3. normalize 'a_cal', 'b_cal', 'a*', 'b*' (min = -120, max = 120) to [0,1]
# สูตร Min-Max Scaling: (X - Min) / (Max - Min)
# ดังนั้น: (X - (-120)) / (120 - (-120)) = (X + 120) / 240
ab_cols = ['a_cal', 'b_cal', 'a*', 'b*']
data[ab_cols] = (data[ab_cols] + 120.0) / 240.0

# ตรวจสอบผลลัพธ์
print(data.describe()) # ดูค่า min, max ของแต่ละคอลัมน์เพื่อความชัวร์

                  R             G             B         L_cal         a_cal  \
count  10090.000000  10090.000000  10090.000000  10090.000000  10090.000000   
mean       0.491852      0.452767      0.508247      0.568066      0.552120   
std        0.329092      0.289262      0.304196      0.219660      0.133455   
min        0.056188      0.000159      0.059925      0.125200      0.280375   
25%        0.130586      0.154259      0.132920      0.395000      0.494458   
50%        0.622724      0.514846      0.685911      0.593800      0.520917   
75%        0.830628      0.684454      0.769218      0.747100      0.674333   
max        0.855434      0.871755      0.886422      0.892200      0.772542   

              b_cal            L*            a*            b*    Crop_Index  
count  10090.000000  10090.000000  10090.000000  10090.000000  10090.000000  
mean       0.492311      0.520746      0.519823      0.515400      3.000000  
std        0.170904      0.218265      0.091017      0

In [21]:
print(data)

              R         G         B   L_cal     a_cal     b_cal      L*  \
0      0.714320  0.001091  0.084396  0.3966  0.769250  0.654708  0.3453   
1      0.713560  0.000409  0.089738  0.3962  0.769458  0.651125  0.3453   
2      0.713959  0.000530  0.095583  0.3966  0.769708  0.647625  0.3453   
3      0.713416  0.014764  0.123118  0.4010  0.766042  0.631917  0.3453   
4      0.711005  0.006379  0.102207  0.3969  0.767333  0.643333  0.3453   
...         ...       ...       ...     ...       ...       ...     ...   
10085  0.846810  0.858575  0.878183  0.8866  0.499625  0.489542  0.9065   
10086  0.833993  0.862299  0.875146  0.8864  0.493208  0.490875  0.9065   
10087  0.845039  0.858431  0.877337  0.8862  0.499000  0.489708  0.9065   
10088  0.845964  0.857840  0.877370  0.8860  0.499583  0.489583  0.9065   
10089  0.844175  0.856236  0.875726  0.8847  0.499542  0.489583  0.9065   

             a*        b*      VDO                 File_Name  Crop_Index  
0      0.631375  0.56754

In [22]:
# สร้างคอลัมน์ 'Base_Color' โดยตัดตัวเลข _1, _2, _3 ด้านหลังออก
# เช่น 'M_136_1' จะกลายเป็น 'M_136'
data['Base_Color'] = data['VDO'].str.rsplit('_', n=1).str[0]


# แบ่ง train test โดย data ที่ column Base_Color มีค่าเดียวกันอยู่ กลุ่มเดียวกัน
# 1. ดึงรายชื่อ Base_Color ทั้งหมดที่ไม่ซ้ำกัน
unique_vdos = data['Base_Color'].unique()

# 2. สับเปลี่ยนลำดับ (Shuffle) รายชื่อ Base_Color เพื่อความสุ่ม
np.random.seed(42)
np.random.shuffle(unique_vdos)

# 3. กำหนดจุดตัดแบ่งข้อมูล (เช่น Train 80%, Test 20%)
split_index = int(len(unique_vdos) * 0.8)

# 4. แบ่งรายชื่อ VDO ออกเป็น 2 กลุ่ม
train_vdo_names = unique_vdos[:split_index]
test_vdo_names = unique_vdos[split_index:]

# 5. กรองข้อมูลจาก DataFrame เดิม
first_data = data[data['Base_Color'].isin(train_vdo_names)].copy()
sec_data = data[data['Base_Color'].isin(test_vdo_names)].copy()

print(f"first_data set: มี {len(first_data)} rows จาก {len(train_vdo_names)} Base_Color")
print(first_data['VDO'].unique())

print(f"sec_data set: มี {len(sec_data)} rows จาก {len(test_vdo_names)} Base_Color")
print(sec_data['VDO'].unique())

first_data set: มี 7660 rows จาก 16 Base_Color
['M_136_1' 'M_136_2' 'M_136_3' 'M_128_1' 'M_128_2' 'M_128_3' 'M_235_1'
 'M_235_2' 'M_235_3' 'M_158_1' 'M_158_2' 'M_158_3' 'M_264_1' 'M_264_2'
 'M_264_3' 'M_266_1' 'M_266_2' 'M_266_3' 'M_327E_1' 'M_327E_2' 'M_327E_3'
 'M_3336_1' 'M_3336_2' 'M_3336_3' 'M_841_1' 'M_841_2' 'M_841_3' 'M_348_1'
 'M_348_2' 'M_348_3' 'M_347_1' 'M_347_2' 'M_347_3' 'M_801_1' 'M_801_2'
 'M_801_3' 'M_814_1' 'M_814_2' 'M_814_3' 'M_504_2' 'M_504_3' 'M_504_1'
 'M_402_1' 'M_402_2' 'M_402_3' 'M_444_1' 'M_444_2' 'M_444_3']
sec_data set: มี 2430 rows จาก 5 Base_Color
['M_324_1' 'M_324_2' 'M_324_3' 'M_327_1' 'M_327_2' 'M_327_3' 'M_835_1'
 'M_835_2' 'M_835_3' 'M_104_1' 'M_104_2' 'M_104_3' 'M_502_1' 'M_502_2'
 'M_502_3']


In [23]:
train_labels = first_data[['L*','a*','b*']]
train_data = first_data.drop(columns=['L*','a*','b*','VDO','File_Name','Crop_Index','Base_Color'])
# สร้าง columns R*G, R*B, G*B, R**2, G**2, B**2
train_data['R*G'] = train_data['R'] * train_data['G']
train_data['R*B'] = train_data['R'] * train_data['B']
train_data['G*B'] = train_data['G'] * train_data['B']

train_data['R**2'] = train_data['R'] ** 2
train_data['G**2'] = train_data['G'] ** 2
train_data['B**2'] = train_data['B'] ** 2

# ตรวจสอบผลลัพธ์
print(train_data.head())

          R         G         B   L_cal     a_cal     b_cal       R*G  \
0  0.714320  0.001091  0.084396  0.3966  0.769250  0.654708  0.000779   
1  0.713560  0.000409  0.089738  0.3962  0.769458  0.651125  0.000292   
2  0.713959  0.000530  0.095583  0.3966  0.769708  0.647625  0.000379   
3  0.713416  0.014764  0.123118  0.4010  0.766042  0.631917  0.010533   
4  0.711005  0.006379  0.102207  0.3969  0.767333  0.643333  0.004535   

        R*B       G*B      R**2          G**2      B**2  
0  0.060286  0.000092  0.510253  1.190238e-06  0.007123  
1  0.064033  0.000037  0.509167  1.676180e-07  0.008053  
2  0.068242  0.000051  0.509737  2.811079e-07  0.009136  
3  0.087834  0.001818  0.508963  2.179618e-04  0.015158  
4  0.072670  0.000652  0.505529  4.068939e-05  0.010446  


In [24]:
test_labels = sec_data[['L*','a*','b*']]
test_data = sec_data.drop(columns=['L*','a*','b*','VDO','File_Name','Crop_Index','Base_Color'])
# สร้าง columns R*G, R*B, G*B, R**2, G**2, B**2
test_data['R*G'] = test_data['R'] * test_data['G']
test_data['R*B'] = test_data['R'] * test_data['B']
test_data['G*B'] = test_data['G'] * test_data['B']

test_data['R**2'] = test_data['R'] ** 2
test_data['G**2'] = test_data['G'] ** 2
test_data['B**2'] = test_data['B'] ** 2

# ตรวจสอบผลลัพธ์
print(test_data.head())

             R         G         B   L_cal     a_cal     b_cal       R*G  \
2930  0.098871  0.296362  0.788805  0.4175  0.609125  0.230083  0.029301   
2931  0.107104  0.309970  0.791004  0.4274  0.601792  0.235583  0.033199   
2932  0.098271  0.295666  0.786782  0.4166  0.608792  0.230625  0.029055   
2933  0.095959  0.297731  0.787397  0.4179  0.607417  0.231125  0.028570   
2934  0.094609  0.295852  0.787396  0.4165  0.608542  0.230208  0.027990   

           R*B       G*B      R**2      G**2      B**2  
2930  0.077990  0.233772  0.009775  0.087830  0.622213  
2931  0.084719  0.245188  0.011471  0.096082  0.625687  
2932  0.077318  0.232625  0.009657  0.087418  0.619026  
2933  0.075558  0.234433  0.009208  0.088644  0.619994  
2934  0.074495  0.232953  0.008951  0.087528  0.619992  


In [25]:
train_data.shape[1]

12

In [26]:
print(len(train_data))
print(len(test_data))

7660
2430


In [27]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

# เลือกเฉพาะคอลัมน์ที่เป็น Feature ของ Quadratic Model (รวม 12 ตัว) ['R', 'G', 'B', 'L_cal', 'a_cal', 'b_cal', 'R*G', 'R*B', 'G*B', 'R**2', 'G**2', 'B**2']

X_train = train_data
y_train = train_labels

X_test = test_data
y_test = test_labels

# สร้างโครงสร้าง Neural Network ตาม Paper
model = Sequential([
    # Input layer รับค่าจาก 12 Quadratic Features
    Input(shape=(X_train.shape[1],)),
    # Hidden layer: ใช้ 2 ชั้นและ 32,16
    Dense(32, activation='relu'),
    Dense(16, activation='relu'),
    # Output layer: 3 นิวรอน สำหรับค่า L*, a*, b* (ใช้ linear activation เพราะเป็น Regression)
    Dense(3, activation='linear')
])

# คอมไพล์โมเดล โดยใช้ loss เป็น Mean Absolute Error (MAE) ตามสมการในเปเปอร์
model.compile(optimizer=Adam(learning_rate=0.001),loss='mae',metrics=['mae', 'mse'])

# สรุปโครงสร้างโมเดล
model.summary()

# ตั้งค่า Early Stopping เพื่อหยุดเทรนเมื่อ Validation Loss ไม่ลดลง (ตามเปเปอร์)
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=100, # หาก error บน validation set ไม่ลดลงติดต่อกัน 40 epochs ให้หยุด
    restore_best_weights=True
)

# เริ่มการเทรนโมเดล
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=500, # ตั้งเผื่อไว้ Early Stopping จะหยุดให้เอง
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

# ประเมินผลลัพธ์กับ Test Set
test_loss, test_mae, test_mse = model.evaluate(X_test, y_test)
print(f"Test MAE: {test_mae:.4f}")

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 32)             │           416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 3)              │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 995 (3.89 KB)

 Trainable params: 995 (3.89 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/500
240/240 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.1115 - mae: 0.1115 - mse: 0.0497 - val_loss: 0.0205 - val_mae: 0.0205 - val_mse: 0.0013
Epoch 2/500
240/240 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.0140 - mae: 0.0140 - mse: 4.9033e-04 - val_loss: 0.0169 - val_mae: 0.0169 - val_mse: 6.5477e-04
Epoch 3/500
240/240 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.0115 - mae: 0.0115 - mse: 3.4301e-04 - val_loss: 0.0140 - val_mae: 0.0140 - val_mse: 3.7116e-04
Epoch 4/500
240/240 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.0104 - mae: 0.0104 - mse: 2.7986e-04 - val_loss: 0.0126 - val_mae: 0.0126 - val_mse: 2.8656e-04
Epoch 5/500
240/240 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.0093 - mae: 0.0093 - mse: 2.4879e-04 - val_loss: 0.0116 - val_mae: 0.0116 - val_mse: 2.6734e-04
Epoch 6/500
240/240 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.0087 - mae: 0.0087 - mse: 2.3808e-04 - val_loss: 0.0113 - val_mae: 0.0113 - val_mse: 2.3976e-04
Epoch 7/500
240/240 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step -

In [28]:
import numpy as np
# model = tf.keras.models.load_model('my_model.keras')
# 1. ทำนายผลลัพธ์จาก Test set ทั้งหมดในรวดเดียว (ไม่ต้องใช้ loop)

X_test = np.vstack((X_train, X_test))
y_test = pd.concat([y_train, y_test], ignore_index=True)

preds_norm = model.predict(X_test)

# 2. Denormalize ผลทำนายและผลจริง กลับเป็นสเกลปกติ
# L* สเกลเดิมคือ 0 ถึง 100 (ตอนแปลงเราหาร 100)
preds_L = preds_norm[:, 0] * 100.0
trues_L = y_test['L*'].values * 100.0

# a* และ b* สเกลเดิมคือ -120 ถึง 120 (ตอนแปลงเราทำ (x+120)/240)
preds_a = (preds_norm[:, 1] * 240.0) - 120.0
trues_a = (y_test['a*'].values * 240.0) - 120.0

preds_b = (preds_norm[:, 2] * 240.0) - 120.0
trues_b = (y_test['b*'].values * 240.0) - 120.0

# นำกลับมารวมเป็น Array เดียวกัน
preds_all = np.column_stack((preds_L, preds_a, preds_b))
trues_all = np.column_stack((trues_L, trues_a, trues_b))

# 3. คำนวณค่า RMSE
rmse_L = np.sqrt(((preds_all[:,0] - trues_all[:,0])**2).mean())
rmse_a = np.sqrt(((preds_all[:,1] - trues_all[:,1])**2).mean())
rmse_b = np.sqrt(((preds_all[:,2] - trues_all[:,2])**2).mean())
rmse_total = np.sqrt(((preds_all - trues_all)**2).mean())

# 4. คำนวณ Error (Mean Normalized Error) ตามสมการใน Paper
e_L = np.abs(preds_all[:,0] - trues_all[:,0]).mean() / 100.0
e_a = np.abs(preds_all[:,1] - trues_all[:,1]).mean() / 240.0
e_b = np.abs(preds_all[:,2] - trues_all[:,2]).mean() / 240.0

# คำนวณ Total Error เป็นเปอร์เซ็นต์
e_total = ((e_L + e_a + e_b) / 3.0) * 100.0

# 5. แสดงผลลัพธ์เปรียบเทียบ
print(f"\nNN(Quad+Lab) RMSE  L:{rmse_L:.2f}  a:{rmse_a:.2f}  b:{rmse_b:.2f}  total:{rmse_total:.2f}")
print(f"NN(Quad+Lab) Error (paper style): {e_total:.2f}%\n")

print("--- เทียบกับผลลัพธ์อ้างอิง ---")
print(f"Paper Quadratic (RGB only):         1.23%") # อ้างอิงจาก Table 3 ใน Paper
print(f"Paper NN (RGB only):                0.93%") # อ้างอิงจาก Table 3 ใน Paper

316/316 ━━━━━━━━━━━━━━━━━━━━ 0s 439us/step

NN(Quad+Lab) RMSE  L:1.21  a:0.97  b:2.04  total:1.48
NN(Quad+Lab) Error (paper style): 0.49%

--- เทียบกับผลลัพธ์อ้างอิง ---
Paper Quadratic (RGB only):         1.23%
Paper NN (RGB only):                0.93%


In [29]:
# Save Model Tensor
# Save the entire model 
model.save('my_model.keras')

In [1]:
# Load the model back later
import tensorflow as tf
loaded_model = tf.keras.models.load_model('my_model.keras')


In [2]:
# import library RGB to Lab
from skimage import color
import numpy as np

def bt709_to_linear(c):
    return np.where(c < 0.081, c / 4.5, ((c + 0.099) / 1.099) ** (1 / 0.45))

def calculate_lab(row):
    r, g, b = row[0] / 255.0, row[1] / 255.0, row[2] / 255.0
    rgb_linear = bt709_to_linear(np.array([r, g, b]))
    # แปลง linear RGB → XYZ ด้วย BT.709/sRGB matrix
    M = np.array([[0.4124564, 0.3575761, 0.1804375],
                  [0.2126729, 0.7151522, 0.0721750],
                  [0.0193339, 0.1191920, 0.9503041]])
    xyz = M @ rgb_linear
    # XYZ → Lab (D65)
    lab_pixel = color.xyz2lab(xyz.reshape(1,1,3), illuminant='D65')
    L, a, b = lab_pixel[0,0]
    return round(L, 2), round(a, 2), round(b, 2)

def prepare_input(rgb_values):
    """
    แปลงค่า RGB เป็น features สำหรับโมเดล (Quadratic + Lab)
    พร้อมปรับ Shape และ Normalize ข้อมูลให้ตรงกับตอน Train
    """
    # 1. รับค่า RGB และ Normalize ให้อยู่ในช่วง [0, 1]
    r = rgb_values[0] / 255.0
    g = rgb_values[1] / 255.0
    b = rgb_values[2] / 255.0
    
    # 2. แปลงเป็น Lab
    L_cal, a_cal, b_cal = calculate_lab(rgb_values)
    
    # 3. *** สำคัญมาก ***: Normalize ค่า Lab ให้เหมือนตอน Train
    L_cal_norm = L_cal / 100.0
    a_cal_norm = (a_cal + 120.0) / 240.0
    b_cal_norm = (b_cal + 120.0) / 240.0
    
    # 4. เรียงลำดับ Features ให้ตรงกับ DataFrame ตอน Train
    # ลำดับใน Train Data: 'R', 'G', 'B', 'L_cal', 'a_cal', 'b_cal', 'R*G', 'R*B', 'G*B', 'R**2', 'G**2', 'B**2'
    features = [
        r, g, b, 
        L_cal_norm, a_cal_norm, b_cal_norm,
        r * g, r * b, g * b,
        r ** 2, g ** 2, b ** 2
    ]
    
    # 5. แปลงเป็น Numpy Array และปรับ Shape เป็น 2D (1, 12) 
    features_array = np.array([features]) 
    
    return features_array

preds_norm = loaded_model.predict(prepare_input([187, 160 , 153]))
preds_L = preds_norm[:, 0] * 100.0
preds_a = (preds_norm[:, 1] * 240.0) - 120.0
preds_b = (preds_norm[:, 2] * 240.0) - 120.0

print(f"Predicted Lab: L*={preds_L[0]:.2f}, a*={preds_a[0]:.2f}, b*={preds_b[0]:.2f}")



1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
Predicted Lab: L*=66.11, a*=6.12, b*=15.46


32-16 : 1.09%